## Imports

In [1]:
import kagglehub
from pathlib import Path
import pandas as pd

c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Téléchargement KaggleHub
Ce bloc télécharge les trois datasets et retrouve automatiquement leurs CSV

In [3]:
def load_first_csv_from_kagglehub(dataset_name: str) -> pd.DataFrame:
    """
    Télécharge un dataset via kagglehub et charge automatiquement
    le premier CSV trouvé dans le dossier.
    """
    path = Path(kagglehub.dataset_download(dataset_name))
    csv_files = list(path.rglob("*.csv"))
    
    if not csv_files:
        raise FileNotFoundError(f"Aucun CSV trouvé dans {path}")
    
    print(f"📥 Dataset '{dataset_name}' chargé depuis : {path}")
    print(f"→ CSV utilisé : {csv_files[0].name}")
    
    return pd.read_csv(csv_files[0]), csv_files[0]

# --- Téléchargement des trois datasets Kaggle ---
df_gym1, file1 = load_first_csv_from_kagglehub("niharika41298/gym-exercise-data")
df_gym2, file2 = load_first_csv_from_kagglehub("rishitmurarka/gym-exercises-dataset")
df_best50, file3 = load_first_csv_from_kagglehub("prajwaldongre/best-50-exercise-for-your-body")

print("Gym Exercise Data:", df_gym1.shape)
print("Gym Exercises Dataset:", df_gym2.shape)
print("Best 50 exercises:", df_best50.shape)

display(df_gym1.head(3))
display(df_gym2.head(3))
display(df_best50.head(3))

📥 Dataset 'niharika41298/gym-exercise-data' chargé depuis : C:\Users\fback\.cache\kagglehub\datasets\niharika41298\gym-exercise-data\versions\1
→ CSV utilisé : megaGymDataset.csv
📥 Dataset 'rishitmurarka/gym-exercises-dataset' chargé depuis : C:\Users\fback\.cache\kagglehub\datasets\rishitmurarka\gym-exercises-dataset\versions\1
→ CSV utilisé : gym_exercise_dataset.csv
📥 Dataset 'prajwaldongre/best-50-exercise-for-your-body' chargé depuis : C:\Users\fback\.cache\kagglehub\datasets\prajwaldongre\best-50-exercise-for-your-body\versions\1
→ CSV utilisé : Top 50 Excerice for your body.csv
Gym Exercise Data: (2918, 9)
Gym Exercises Dataset: (617, 17)
Best 50 exercises: (50, 8)


,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


,Exercise Name,Equipment,Variation,Utility,Mechanics,Force,Preparation,Execution,Target_Muscles,Synergist_Muscles,Stabilizer_Muscles,Antagonist_Muscles,Dynamic_Stabilizer_Muscles,Main_muscle,Difficulty (1-5),Secondary Muscles,parent_id
0,Neck Flexion,Cable,No,Basic or Auxiliary,Isolated,Pull,Sit on bench facing away from middle pulley. P...,Move head away from pulley by bending neck for...,"Sternocleidomastoid,","None,","Rectus Abdominis, Obliques,",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
1,Neck Flexion,Lever (plate loaded),No,Basic or Auxiliary,Isolated,Pull,Sit on seat in machine. Position padded lever ...,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,","None,","Latissimus Dorsi, Deltoid, Posterior, Rhomboid...",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
2,Lateral Neck Flexion,Lever (plate loaded),No,Auxiliary,Isolated,Pull,Sit on seat in machine with feet apart . Pos...,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,","Splenius, Erector Spinae, Levator Scapulae, Tr...","Latissimus Dorsi, Pectoralis Major, Sternal, P...",NaN,NaN,Neck,2,"Sternocleidomastoid, Levator Scapulae",NaN


,Name of Exercise,Sets,Reps,Benefit,Burns Calories (per 30 min),Target Muscle Group,Equipment Needed,Difficulty Level
0,Push-ups,3,15,Builds upper body strength,200,"Chest, Triceps, Shoulders",NaN,Intermediate
1,Squats,4,12,Strengthens lower body,223,"Quadriceps, Hamstrings, Glutes",NaN,Beginner
2,Lunges,3,10,Improves balance and coordination,275,"Quadriceps, Hamstrings, Glutes",NaN,Beginner


## Harmoniser les colonnes (mapping de la capture Nicolas)

On se crée un schéma commun

In [4]:
CANONICAL_COLS = [
    "exercise_name",       # Nom de l'exercice
    "execution",           # Description / Execution
    "target_muscles",      # BodyPart / Target_Muscles / Target muscle Group
    "equipment",           # Equipment / Equipment needed
    "difficulty",          # Level / Difficulty level
    "source_dataset",      # d'où vient la ligne
]


### Dataset Gym_Exercise_DataSet (niharika41298)

D’après la capture :
- Nom de l'exo → nom
- Description → execution
- BodyPart → target_muscles
- Equipment → equipment
- Level → difficulty

In [10]:
df1 = pd.DataFrame({
    "exercise_name":  df_gym1["Title"],
    "execution":      df_gym1["Desc"],
    "target_muscles": df_gym1["BodyPart"],
    "equipment":      df_gym1["Equipment"],
    "difficulty":     df_gym1["Level"],
})
df1["source_dataset"] = "gym_exercise_data"
df1.head()

,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Partner plank band row,The partner plank band row is an abdominal exe...,Abdominals,Bands,Intermediate,gym_exercise_data
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Abdominals,Bands,Intermediate,gym_exercise_data
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Abdominals,Bands,Intermediate,gym_exercise_data
3,Banded crunch,The banded crunch is an exercise targeting the...,Abdominals,Bands,Intermediate,gym_exercise_data
4,Crunch,The crunch is a popular core exercise targetin...,Abdominals,Bands,Intermediate,gym_exercise_data


### Dataset Gym Exercices (rishitmurarka)

D’après la capture :

- Nom de l'exercice → nom
- Execution → execution
- Target_Muscles → target_muscles
- Equipement → equipment

(s’il y a Difficulty ou Level, on le branche sur difficulty)

In [13]:
df2 = pd.DataFrame({
    "exercise_name":  df_gym2["Exercise Name"],
    "execution":      df_gym2["Execution"],
    "target_muscles": df_gym2["Target_Muscles"],
    "equipment":      df_gym2["Equipment"],
    # adapte le nom exact de la colonne de niveau :
    "difficulty":     df_gym2.get("Difficulty", pd.NA),
})
df2["source_dataset"] = "gym_exercises_dataset"
df2.head()

,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Neck Flexion,Move head away from pulley by bending neck for...,"Sternocleidomastoid,",Cable,<NA>,gym_exercises_dataset
1,Neck Flexion,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,",Lever (plate loaded),<NA>,gym_exercises_dataset
2,Lateral Neck Flexion,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,",Lever (plate loaded),<NA>,gym_exercises_dataset
3,Neck Flexion,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,",Lever (selectorized),<NA>,gym_exercises_dataset
4,Lateral Neck Flexion,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,",Lever (selectorized),<NA>,gym_exercises_dataset


In [ ]:
df3 = pd.DataFrame({
    "exercise_name":  df_best50["Name of Exercise"],
    # ce dataset n'a peut-être pas de vraie description → NaN
    "execution":      pd.NA,
    "target_muscles": df_best50["Target muscle Group"],
    "equipment":      df_best50["Equipment needed"],
    "difficulty":     df_best50["Difficulty level"],
})
df3["source_dataset"] = "best_50_exercises"
df3.head()